<a href="https://colab.research.google.com/github/Avjohana/ChemChettos/blob/main/ChemHack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q numpy pandas scipy soundfile librosa praat-parselmouth
!pip install -q xgboost scikit-learn joblib transformers accelerate
!pip install -q faster-whisper ctranslate2 fastapi uvicorn nest-asyncio requests

import os, glob, shutil, time, json, base64, tempfile, gc
import numpy as np
import pandas as pd

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

PROJECT_DIR = '/content/drive/MyDrive/Audios'
AUDIO_DIR_SRC = f'{PROJECT_DIR}/all_audio'
LOCAL_AUDIO = '/content/audio_local'
MODELS_DIR  = f'{PROJECT_DIR}/models'
FEAT_DIR    = f'{PROJECT_DIR}/features_cache'

USE_PARSELMOUTH = False
WHISPER_SIZE    = "small"
WAV2VEC_MAX_SEC = 90
DSP_MAX_SEC     = 90
N_JOBS          = 2
FORCE_RETRAIN   = False

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FEAT_DIR, exist_ok=True)

manifest_src = None
for c in [f'{PROJECT_DIR}/manifest.csv', f'{PROJECT_DIR}/data/manifest.csv']:
    if os.path.exists(c):
        manifest_src = c
        break
if manifest_src is None:
    hits = glob.glob('/content/drive/MyDrive/**/manifest*.csv', recursive=True)
    manifest_src = hits[0] if hits else None
if manifest_src is None:
    raise FileNotFoundError("manifest.csv no encontrado")

manifest = pd.read_csv(manifest_src)
if 'anon_id' in manifest.columns and 'call_id' not in manifest.columns:
    manifest = manifest.rename(columns={'anon_id': 'call_id'})
if manifest['label'].dtype == object:
    manifest['label'] = manifest['label'].map({'human': 0, 'synthetic': 1})
print(f"Manifest: {len(manifest)} filas - {manifest['label'].value_counts().to_dict()}")

if not os.path.exists(LOCAL_AUDIO) or len(os.listdir(LOCAL_AUDIO)) < 100:
    print("Copiando audios...")
    os.makedirs(LOCAL_AUDIO, exist_ok=True)
    t0 = time.time()
    sources = glob.glob(f'{AUDIO_DIR_SRC}/**/*.wav', recursive=True)
    if not sources:
        sources = glob.glob(f'{PROJECT_DIR}/**/*.wav', recursive=True)
    for w in sources:
        dst = os.path.join(LOCAL_AUDIO, os.path.basename(w))
        if not os.path.exists(dst):
            try:
                shutil.copy2(w, dst)
            except Exception:
                pass
    print(f"Copiados {len(os.listdir(LOCAL_AUDIO))} audios en {time.time()-t0:.1f}s")
else:
    print(f"Audios en local: {len(os.listdir(LOCAL_AUDIO))}")

AUDIO_DIR = LOCAL_AUDIO

import librosa
import torch
import xgboost as xgb
import joblib
from scipy.stats import kurtosis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, confusion_matrix
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
from faster_whisper import WhisperModel
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
import nest_asyncio
import uvicorn
from threading import Thread
from joblib import Parallel, delayed

nest_asyncio.apply()


def extract_dsp_features(audio_path, sr=8000):
    y, _ = librosa.load(audio_path, sr=sr, mono=False)
    if y.ndim > 1:
        y = y[0]
    y = y[:DSP_MAX_SEC * sr]
    feats = []
    win = 3 * sr
    hop = win
    for start in range(0, len(y) - win + 1, hop):
        w = y[start:start + win]
        if np.sqrt(np.mean(w**2)) < 1e-4:
            continue
        try:
            mfcc = librosa.feature.mfcc(y=w, sr=sr, n_mfcc=13)
            var_mfcc = np.var(librosa.feature.delta(mfcc))
            var_flat = np.var(librosa.feature.spectral_flatness(y=w))
            stft = np.abs(librosa.stft(w))
            freqs = librosa.fft_frequencies(sr=sr)
            hf = np.sum(stft[freqs > 6000])
            lf = np.sum(stft[(freqs >= 300) & (freqs <= 4000)])
            ratio = hf / (lf + 1e-10)
            var_cent = np.var(librosa.feature.spectral_centroid(y=w, sr=sr))
            try:
                f0 = librosa.yin(w, fmin=60, fmax=400, sr=sr)
                f0_clean = f0[f0 > 0]
                if len(f0_clean) > 5:
                    diffs = np.abs(np.diff(f0_clean))
                    jit = np.mean(diffs) / (np.mean(f0_clean) + 1e-10)
                else:
                    jit = 0.0
                env = np.abs(w)
                env_smooth = np.convolve(env, np.ones(80)/80, mode='same')
                shim = np.mean(np.abs(np.diff(env_smooth))) / (np.mean(env_smooth) + 1e-10)
            except Exception:
                jit, shim = 0.0, 0.0
            feats.append([var_mfcc, var_flat, ratio, var_cent, jit, shim])
        except Exception:
            continue
    return np.mean(feats, axis=0) if feats else np.zeros(6)


def extract_turns_from_audio(audio_path, sr=8000, frame_ms=30, energy_thresh=0.01):
    y, _ = librosa.load(audio_path, sr=sr, mono=False)
    if y.ndim == 1:
        return []
    caller, agent = y[0], y[1]
    frame_len = int(sr * frame_ms / 1000)
    turns = []
    for ch_name, ch in [('caller', caller), ('agent', agent)]:
        if len(ch) < frame_len:
            continue
        n = len(ch) // frame_len
        ch_trim = ch[:n*frame_len].reshape(n, frame_len)
        energy = np.sum(ch_trim ** 2, axis=1)
        if energy.max() > 0:
            thresh = max(energy_thresh, 0.05 * energy.max())
        else:
            thresh = energy_thresh
        is_sp = energy > thresh
        changes = np.diff(is_sp.astype(np.int8))
        starts = np.where(changes == 1)[0] + 1
        ends = np.where(changes == -1)[0] + 1
        if is_sp[0]:
            starts = np.concatenate([[0], starts])
        if is_sp[-1]:
            ends = np.concatenate([ends, [n]])
        for s, e in zip(starts, ends):
            turns.append({'speaker': ch_name,
                          'start': s * frame_ms / 1000,
                          'end': e * frame_ms / 1000})
    turns.sort(key=lambda t: t['start'])
    return turns


def extract_conversational_features(audio_path, sr=8000):
    try:
        turns = extract_turns_from_audio(audio_path, sr)
    except Exception:
        return np.array([0.0, 0.0, 0.0, 0.0])
    ct = [t for t in turns if t['speaker'] == 'caller']
    at = [t for t in turns if t['speaker'] == 'agent']
    if len(ct) < 2 or len(at) < 1:
        return np.array([0.0, 0.0, 0.0, 0.0])
    rd = []
    for a in at:
        nxt = [c for c in ct if c['start'] > a['end']]
        if nxt:
            rd.append(nxt[0]['start'] - a['end'])
    rd = np.array(rd)
    if len(rd) < 2:
        return np.array([0.0, 0.0, 0.0, 0.0])
    cv = np.std(rd) / (np.mean(rd) + 1e-10)
    ku = kurtosis(rd)
    total = max(t['end'] for t in turns) - min(t['start'] for t in turns)
    ov = 0.0
    for c in ct:
        for a in at:
            s, e = max(c['start'], a['start']), min(c['end'], a['end'])
            if e > s:
                ov += (e - s)
    ov_d = ov / (total + 1e-10)
    short = [c for c in ct if c['end'] - c['start'] < 0.5]
    bc = len(short) / (total + 1e-10)
    return np.array([cv, ku, ov_d, bc])


def compute_p_C(texts):
    negs = ['no', 'nunca', 'no tengo', 'no sé', 'no conozco', 'no me acuerdo']
    nn, nw = 0, 0
    for t in texts:
        nw += len(t.lower().split())
        for n in negs:
            if n in t.lower():
                nn += 1
    if nw == 0:
        return 0.5
    return 1.0 - min((nn / nw) * 10, 1.0)


def fuse_predictions(p_A, p_B, p_C=None):
    if p_C is None:
        return float(np.clip(0.6 * p_A + 0.4 * p_B, 0.0, 1.0))
    return float(np.clip(0.5 * p_A + 0.3 * p_B + 0.2 * p_C, 0.0, 1.0))


DSP_CACHE = f'{FEAT_DIR}/dsp_features.npz'
CONV_CACHE = f'{FEAT_DIR}/conv_features.npz'

audio_index = {}
for w in glob.glob(f'{AUDIO_DIR}/**/*.wav', recursive=True):
    stem = os.path.splitext(os.path.basename(w))[0]
    audio_index[stem] = w
    audio_index[stem.replace('call_', '').replace('anon_', '')] = w

paths, keys = [], []
for cid in manifest['call_id']:
    ap = audio_index.get(cid) or audio_index.get(cid.replace('call_', '').replace('anon_', ''))
    if ap:
        paths.append(ap)
        keys.append(cid)

print(f"Audios indexados: {len(paths)}")

if os.path.exists(DSP_CACHE) and not FORCE_RETRAIN:
    data = np.load(DSP_CACHE, allow_pickle=True)
    dsp_feats = {k: data[k] for k in data.files}
    print(f"DSP features del cache: {len(dsp_feats)}")
else:
    print(f"Calculando DSP features (n_jobs={N_JOBS})...")
    t0 = time.time()
    results = Parallel(n_jobs=N_JOBS, verbose=1, batch_size=8)(
        delayed(extract_dsp_features)(p) for p in paths
    )
    dsp_feats = {k: r for k, r in zip(keys, results)}
    print(f"{len(dsp_feats)} en {time.time()-t0:.1f}s")
    np.savez_compressed(DSP_CACHE, **dsp_feats)

if os.path.exists(CONV_CACHE) and not FORCE_RETRAIN:
    data = np.load(CONV_CACHE, allow_pickle=True)
    conv_feats = {k: data[k] for k in data.files}
    print(f"Features conversacionales del cache: {len(conv_feats)}")
else:
    print(f"Calculando features conversacionales (n_jobs={N_JOBS})...")
    t0 = time.time()
    results = Parallel(n_jobs=N_JOBS, verbose=1, batch_size=8)(
        delayed(extract_conversational_features)(p) for p in paths
    )
    conv_feats = {k: r for k, r in zip(keys, results)}
    print(f"{len(conv_feats)} en {time.time()-t0:.1f}s")
    np.savez_compressed(CONV_CACHE, **conv_feats)


DSP_PATH = f'{MODELS_DIR}/dsp_classifier.pkl'
XGB_PATH = f'{MODELS_DIR}/xgb_conversational.pkl'

if os.path.exists(DSP_PATH) and not FORCE_RETRAIN:
    print("Cargando DSP classifier...")
    dsp_clf = joblib.load(DSP_PATH)
else:
    print("Entrenando DSP classifier...")
    Xd, yd = [], []
    for _, row in manifest[manifest['split'] == 'train'].iterrows():
        cid = row['call_id']
        if cid in dsp_feats:
            Xd.append(dsp_feats[cid])
            yd.append(int(row['label']))
    Xd = np.array(Xd)
    yd = np.array(yd)
    print(f"DSP train: {Xd.shape} | labels: {np.bincount(yd)}")
    dsp_clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    dsp_clf.fit(Xd, yd)
    joblib.dump(dsp_clf, DSP_PATH)

if os.path.exists(XGB_PATH) and not FORCE_RETRAIN:
    print("Cargando XGBoost...")
    xgb_model = joblib.load(XGB_PATH)
else:
    print("Entrenando XGBoost...")
    X, y = [], []
    for _, row in manifest.iterrows():
        cid = row['call_id']
        if cid in conv_feats:
            X.append(conv_feats[cid])
            y.append(int(row['label']))
    X = np.array(X, dtype=np.float64)
    y = np.array(y, dtype=np.int64)
    print(f"Features: {X.shape} | labels: {np.bincount(y)}")
    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, eval_metric='logloss')
    xgb_model.fit(X_tr, y_tr)
    acc = balanced_accuracy_score(y_va, xgb_model.predict(X_va))
    print(f"Balanced accuracy (val): {acc:.4f}")
    joblib.dump(xgb_model, XGB_PATH)


gc.collect()

print("Cargando Wav2Vec2...")
wav2vec_model = AutoModelForAudioClassification.from_pretrained("garystafford/wav2vec2-deepfake-voice-detector")
feat_ext = AutoFeatureExtractor.from_pretrained("garystafford/wav2vec2-deepfake-voice-detector")
device = "cuda" if torch.cuda.is_available() else "cpu"
wav2vec_model.to(device).eval()
print(f"Wav2Vec2 en {device}")
!free -h

print(f"Cargando Whisper {WHISPER_SIZE}...")
whisper_model = WhisperModel(WHISPER_SIZE,
                              device="cuda" if torch.cuda.is_available() else "cpu",
                              compute_type="float16" if torch.cuda.is_available() else "int8")
print("Whisper cargado")
!free -h


def predict_wav2vec2(audio_path, sr=8000):
    y, _ = librosa.load(audio_path, sr=sr, mono=True)
    y = y[:WAV2VEC_MAX_SEC * sr]
    y16 = librosa.resample(y, orig_sr=sr, target_sr=16000)
    inputs = feat_ext(y16, sampling_rate=16000, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = wav2vec_model(**inputs)
        probs = torch.nn.functional.softmax(out.logits, dim=-1)
    return probs[0][1].item()


app = FastAPI()

class DetectRequest(BaseModel):
    call_id: str
    audio_base64: str
    sample_rate: int = 8000
    channels: int = 2


@app.post("/detect")
async def detect(req: DetectRequest):
    try:
        audio_bytes = base64.b64decode(req.audio_base64)
    except Exception:
        return {"is_synthetic": False, "confidence": 0.5}

    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp.write(audio_bytes)
        tmp_path = tmp.name

    try:
        try:
            dsp_f = extract_dsp_features(tmp_path)
            p_A1 = float(dsp_clf.predict_proba([dsp_f])[0][1])
        except Exception:
            p_A1 = 0.5

        try:
            p_A2 = predict_wav2vec2(tmp_path)
        except Exception:
            p_A2 = 0.5

        p_A = 0.3 * p_A1 + 0.7 * p_A2

        try:
            feats_B = extract_conversational_features(tmp_path)
            p_B = float(xgb_model.predict_proba([feats_B])[0][1])
        except Exception:
            p_B = 0.5

        p_C = None
        if abs(p_A - p_B) > 0.4:
            try:
                turns = extract_turns_from_audio(tmp_path)
                segs = [t for t in turns if t['speaker'] == 'caller']
                ya, _ = librosa.load(tmp_path, sr=8000, mono=False)
                if ya.ndim > 1:
                    ya = ya[0]
                texts = []
                for t in segs:
                    s, e = int(t['start']*8000), int(t['end']*8000)
                    seg = ya[s:e]
                    if len(seg) < 800:
                        continue
                    sg, _ = whisper_model.transcribe(seg, language="es")
                    texts.append(" ".join(x.text for x in sg))
                p_C = compute_p_C(texts)
            except Exception:
                p_C = None

        p_final = fuse_predictions(p_A, p_B, p_C)
        return {"is_synthetic": bool(p_final > 0.5), "confidence": float(p_final)}
    finally:
        try:
            os.unlink(tmp_path)
        except Exception:
            pass
        gc.collect()


@app.get("/health")
async def health():
    return {"status": "ok"}


HTML_PAGE = """<!DOCTYPE html>
<html lang="es"><head><meta charset="UTF-8"><title>Madre</title>
<style>
body{font-family:-apple-system,sans-serif;background:#0f172a;color:#e2e8f0;margin:0;padding:40px;min-height:100vh}
.container{max-width:720px;margin:0 auto}
h1{font-size:28px;margin-bottom:8px;color:#38bdf8}
.subtitle{color:#94a3b8;margin-bottom:32px;font-size:14px}
.card{background:#1e293b;border-radius:12px;padding:24px;margin-bottom:20px;border:1px solid #334155}
label{display:block;margin-bottom:8px;font-size:14px;color:#94a3b8}
input[type="file"]{width:100%;padding:12px;background:#0f172a;color:#e2e8f0;border:1px dashed #475569;border-radius:8px;cursor:pointer}
button{background:#38bdf8;color:#0f172a;border:none;padding:12px 24px;border-radius:8px;font-weight:600;cursor:pointer;font-size:14px;margin-top:16px}
button:disabled{opacity:0.5;cursor:not-allowed}
.result{margin-top:20px;padding:20px;border-radius:8px;display:none}
.result.human{background:#064e3b;border-left:4px solid #10b981}
.result.synth{background:#7f1d1d;border-left:4px solid #ef4444}
.result.error{background:#78350f;border-left:4px solid #f59e0b}
.verdict{font-size:24px;font-weight:700;margin-bottom:12px}
.metric{display:flex;justify-content:space-between;padding:6px 0;font-size:14px;border-bottom:1px solid #1e293b}
.metric span:first-child{color:#94a3b8}
.bar{height:8px;background:#0f172a;border-radius:4px;overflow:hidden;margin-top:8px}
.bar-fill{height:100%;background:linear-gradient(90deg,#10b981,#ef4444)}
.loading{color:#94a3b8;font-style:italic}
</style></head><body>
<div class="container">
<h1>Madre</h1>
<div class="subtitle">Detector de voz sintetica</div>
<div class="card">
<label>WAV (estereo 8 kHz, canal 0 = llamante)</label>
<input type="file" id="wavInput" accept=".wav,audio/wav">
<button id="analyzeBtn" onclick="analyze()">Analizar</button>
</div>
<div id="result" class="result"></div>
</div>
<script>
async function analyze(){
  const fi=document.getElementById('wavInput');
  const rd=document.getElementById('result');
  const btn=document.getElementById('analyzeBtn');
  if(!fi.files.length){rd.className='result error';rd.style.display='block';rd.innerHTML='<div class="verdict">Selecciona un archivo</div>';return;}
  const file=fi.files[0];
  btn.disabled=true;rd.className='result';rd.style.display='block';rd.innerHTML='<div class="loading">Analizando...</div>';
  try{
    const ab=await file.arrayBuffer();
    const bytes=new Uint8Array(ab);let bin='';
    for(let i=0;i<bytes.length;i++)bin+=String.fromCharCode(bytes[i]);
    const b64=btoa(bin);
    const t0=performance.now();
    const res=await fetch('/detect',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({call_id:file.name.replace('.wav',''),audio_base64:b64,sample_rate:8000,channels:2})});
    const t1=performance.now();
    if(!res.ok){rd.className='result error';rd.innerHTML='<div class="verdict">Error HTTP '+res.status+'</div>';return;}
    const data=await res.json();
    const isSynth=data.is_synthetic,conf=data.confidence??0.5,lat=(t1-t0).toFixed(0);
    rd.className='result '+(isSynth?'synth':'human');
    rd.innerHTML='<div class="verdict">'+(isSynth?'VOZ SINTETICA':'VOZ HUMANA')+'</div>'+
      '<div class="metric"><span>Confianza</span><span>'+(conf*100).toFixed(1)+'%</span></div>'+
      '<div class="bar"><div class="bar-fill" style="width:'+(conf*100)+'%"></div></div>'+
      '<div class="metric" style="margin-top:12px"><span>Latencia</span><span>'+lat+' ms</span></div>'+
      '<div class="metric"><span>Archivo</span><span>'+file.name+'</span></div>';
  }catch(e){rd.className='result error';rd.innerHTML='<div class="verdict">Error</div><div>'+e.message+'</div>';}
  finally{btn.disabled=false;}
}
</script></body></html>"""

@app.get("/", response_class=HTMLResponse)
async def index():
    return HTML_PAGE


_server = None
_server_thread = None

def start_server(port=8000):
    global _server, _server_thread
    if _server is not None:
        return
    cfg = uvicorn.Config(app, host="0.0.0.0", port=port, log_level="warning")
    _server = uvicorn.Server(cfg)
    _server_thread = Thread(target=_server.run, daemon=True)
    _server_thread.start()
    time.sleep(3)
    print(f"Servidor en puerto {port}")

def stop_server():
    global _server, _server_thread
    if _server is not None:
        _server.should_exit = True
        time.sleep(1)
        _server = None
        _server_thread = None

def evaluar(split='val', n=20, timeout=120):
    import requests
    subset = manifest[manifest['split'] == split].head(n)
    print(f"\nEvaluando {len(subset)} llamadas del split '{split}'")
    res = []
    t0 = time.time()
    for i, row in subset.iterrows():
        cid = row['call_id']
        yt = int(row['label'])
        ap = os.path.join(AUDIO_DIR, f'{cid}.wav')
        if not os.path.exists(ap):
            hits = glob.glob(f'{AUDIO_DIR}/**/{cid}.wav', recursive=True)
            if not hits:
                continue
            ap = hits[0]
        with open(ap, 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        s = time.time()
        try:
            r = requests.post("http://localhost:8000/detect",
                              json={"call_id": cid, "audio_base64": b64,
                                    "sample_rate": 8000, "channels": 2},
                              timeout=timeout)
            lat = time.time() - s
            if r.status_code != 200:
                continue
            d = r.json()
            if not isinstance(d.get('is_synthetic'), bool):
                continue
            yp = int(d['is_synthetic'])
            cf = float(d.get('confidence', 0.5))
            mk = "ok" if yp == yt else "X"
            print(f"  [{i+1}/{len(subset)}] real={yt} pred={yp} conf={cf:.3f} lat={lat:.2f}s {mk}")
            res.append((cid, yt, yp, cf, lat))
        except Exception as e:
            print(f"  [{i+1}] ERROR {e}")

    if res:
        yt = np.array([r[1] for r in res])
        yp = np.array([r[2] for r in res])
        cf = np.array([r[3] for r in res])
        lt = np.array([r[4] for r in res])
        print(f"\nBalanced accuracy: {balanced_accuracy_score(yt, yp):.4f}")
        print(f"Latencia: media={lt.mean():.2f}s | p95={np.percentile(lt,95):.2f}s")
    return res


!fuser -k 8000/tcp 2>/dev/null
time.sleep(2)
start_server(8000)

Manifest: 362 filas - {1: 257, 0: 105}
Audios en local: 308
Audios indexados: 362
DSP features del cache: 308
Features conversacionales del cache: 308
Cargando DSP classifier...
Cargando XGBoost...
Cargando Wav2Vec2...


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Wav2Vec2 en cuda
               total        used        free      shared  buff/cache   available
Mem:            12Gi       4.8Gi       491Mi        20Mi       7.7Gi       7.8Gi
Swap:             0B          0B          0B
Cargando Whisper small...
Whisper cargado
               total        used        free      shared  buff/cache   available
Mem:            12Gi       4.8Gi       491Mi        20Mi       7.7Gi       7.8Gi
Swap:             0B          0B          0B


In [ ]:
!pip install -q gradio requests matplotlib

import gradio as gr
import requests
import base64
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt


API_URL = "http://localhost:8000/detect"


def analyze_audio(audio_path):
    if audio_path is None:
        return None, "Seleccione un archivo WAV.", ""

    with open(audio_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()

    try:
        r = requests.post(
            API_URL,
            json={
                "call_id": os.path.basename(audio_path),
                "audio_base64": b64,
                "sample_rate": 8000,
                "channels": 2
            },
            timeout=60
        )
        r.raise_for_status()
        d = r.json()
    except Exception as e:
        return None, f"Error de conexion: {e}", ""

    is_synthetic = d.get("is_synthetic", False)
    confidence = d.get("confidence", 0.5)

    if is_synthetic:
        verdict_html = (
            "<div style='padding:28px; border-radius:14px; background:#7f1d1d; "
            "border-left:8px solid #ef4444; text-align:center;'>"
            "<div style='font-size:40px; font-weight:900; color:#fecaca; "
            "letter-spacing:3px; margin-bottom:10px;'>VOZ SINTETICA</div>"
            "<div style='font-size:20px; color:#fca5a5;'>"
            f"Confianza: {confidence*100:.1f}%</div></div>"
        )
    else:
        verdict_html = (
            "<div style='padding:28px; border-radius:14px; background:#064e3b; "
            "border-left:8px solid #10b981; text-align:center;'>"
            "<div style='font-size:40px; font-weight:900; color:#a7f3d0; "
            "letter-spacing:3px; margin-bottom:10px;'>VOZ HUMANA</div>"
            "<div style='font-size:20px; color:#6ee7b7;'>"
            f"Confianza: {confidence*100:.1f}%</div></div>"
        )

    details = (
        f"Archivo: {os.path.basename(audio_path)}\n"
        f"Veredicto: {'sintetico' if is_synthetic else 'humano'}\n"
        f"Confianza: {confidence:.4f}"
    )

    try:
        y, sr = librosa.load(audio_path, sr=8000, mono=False)
        if y.ndim == 1:
            y = np.stack([y, y])
        fig, axes = plt.subplots(2, 1, figsize=(9, 3.5), sharex=True)
        t = np.arange(y.shape[1]) / sr
        axes[0].plot(t, y[0], color='#1f77b4', linewidth=0.6)
        axes[0].set_ylabel('Canal 0\n(llamante)')
        axes[1].plot(t, y[1], color='#ff7f0e', linewidth=0.6)
        axes[1].set_ylabel('Canal 1\n(agente)')
        axes[1].set_xlabel('Tiempo (s)')
        for ax in axes:
            ax.grid(True, alpha=0.3)
        plt.tight_layout()
    except Exception:
        fig = None

    return fig, verdict_html, details


CUSTOM_CSS = """
#verdict-box .prose { width: 100%; }
#verdict-box { padding: 0 !important; }
"""

with gr.Blocks(title="ChemHack", css=CUSTOM_CSS, theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# ChemHack\n"
        "### Detector de voz sintetica en llamadas telefonicas\n"
        "Cargue un archivo WAV estereo (8 kHz). Canal 0: llamante. Canal 1: agente."
    )

    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(label="Archivo de audio", type="filepath",
                                   sources=["upload"])
            analyze_btn = gr.Button("Analizar", variant="primary", size="lg")
        with gr.Column(scale=2):
            verdict_output = gr.HTML(label="Veredicto", elem_id="verdict-box")
            plot_output = gr.Plot(label="Forma de onda")
            info_output = gr.Textbox(label="Detalles", lines=4)

    analyze_btn.click(
        fn=analyze_audio,
        inputs=audio_input,
        outputs=[plot_output, verdict_output, info_output]
    )

demo.launch(share=True)